### 결측치 & 이상치 체크

In [1]:
import pandas as pd

da = pd.read_csv('../data/interim/dailyActivity_merged.csv')
da['ActivityDate'] = pd.to_datetime(da['ActivityDate'], format='%m/%d/%Y')

In [2]:
na = da.isna().sum()
print(na[na > 0] if na.sum() else '결측 없음')

결측 없음


In [3]:
print('TotalSteps==0:', (da['TotalSteps']==0).sum(), '행')
print('Calories==0:', (da['Calories']==0).sum(), '행')
print('TotalSteps==0 이면서 Calories>0:', ((da['TotalSteps']==0)&(da['Calories']>0)).sum())

print()

print('30,000걸음 이상:', (da['TotalSteps']>=30000).sum(), '행')
print(da['TotalSteps'].describe())

TotalSteps==0: 138 행
Calories==0: 9 행
TotalSteps==0 이면서 Calories>0: 129

30,000걸음 이상: 1 행
count     1397.000000
mean      7280.898354
std       5214.336113
min          0.000000
25%       3146.000000
50%       6999.000000
75%      10544.000000
max      36019.000000
Name: TotalSteps, dtype: float64


In [4]:
print('Calories < 500 (미착용 의심):', (da['Calories']<500).sum(), '행')

print()

print('Sedentary 24시간(1440분) 기록 (완전 미착용 의심일)')
print((da['SedentaryMinutes']>=1440).sum(), '행 /', len(da), '행 (약 {:.1f}%)'.format(
    100*(da['SedentaryMinutes']>=1440).sum()/len(da)))


Calories < 500 (미착용 의심): 21 행

Sedentary 24시간(1440분) 기록 (완전 미착용 의심일)
142 행 / 1397 행 (약 10.2%)


### sleepDay 중복 확인

In [5]:
sleep = pd.read_csv('../data/raw/02/sleepDay_merged.csv')
sleep['SleepDay'] = pd.to_datetime(sleep['SleepDay'])
dup = sleep.groupby(['Id','SleepDay']).size()

print('하루에 sleep 기록이 2건 이상인 경우:', (dup>1).sum(), '건')
print(dup[dup>1])


하루에 sleep 기록이 2건 이상인 경우: 3 건
Id          SleepDay  
4388161847  2016-05-05    2
4702921684  2016-05-07    2
8378563200  2016-04-25    2
dtype: int64


C:\Users\color\AppData\Local\Temp\ipykernel_15952\1385009468.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sleep['SleepDay'] = pd.to_datetime(sleep['SleepDay'])


**정제 규칙**
1. `SedentaryMinutes >= 1440` → 미착용일 플래그
2. `Calories == 0` → 미착용일 플래그
3. sleepDay 완전 중복행 제거 후 병합
4. 걸음수 상한 이상치는 이번 데이터에선 미미해 별도 제거하지 않음